In [ ]:
# Cell 1: Install Required Packages
# Run this cell first - may take 2-3 minutes
# After installation completes, restart kernel and run this cell again

import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

# Install packages using the current Python interpreter
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install lancedb>=0.4.0
!{sys.executable} -m pip install torch>=2.0.0 torchvision>=0.15.0
!{sys.executable} -m pip install transformers>=4.35.0
!{sys.executable} -m pip install Pillow>=10.0.0
!{sys.executable} -m pip install tqdm>=4.66.0
!{sys.executable} -m pip install requests>=2.31.0
!{sys.executable} -m pip install numpy>=1.24.0
!{sys.executable} -m pip install pandas>=2.0.0
!{sys.executable} -m pip install pyarrow>=14.0.0
!{sys.executable} -m pip install pydantic>=2.0.0

print("\n" + "=" * 60)
print("✓ All packages installed!")
print("=" * 60)
print("\n⚠️ If this is the first run, restart the kernel now,")
print("   then run this cell again, followed by Cell 2.")

In [ ]:
# Cell 2: Import Libraries & Check GPU

import os
import json
import time
import pickle
import requests
import zipfile
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import ViltProcessor, ViltModel
import lancedb
import pyarrow as pa

# Check GPU availability
print("=" * 60)
print("GPU Configuration")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
else:
    print("⚠️ No GPU detected! Embedding generation will be slower.")
    DEVICE = torch.device("cpu")

print(f"\n✓ Using device: {DEVICE}")
print(f"✓ LanceDB version: {lancedb.__version__}")

In [ ]:
# Cell 3: Configuration Settings

# ============================================
# 🔧 CONFIGURATION - MODIFY THESE VALUES
# ============================================

# LanceDB Storage - Using BOTH local and remote!
import platform
import os

# Detect Azure/cloud environment and use persistent storage
if platform.system() == 'Linux':
    # Check for persistent writable locations (in order of preference)
    if os.path.exists('/home/azureuser') and os.access('/home/azureuser', os.W_OK):
        # Azure ML - use /home/azureuser which is PERSISTENT and supports ext4
        LANCEDB_LOCAL_URI = "/home/azureuser/lancedb_data"
        print("☁️ Azure ML detected - using /home/azureuser for PERSISTENT local LanceDB")
    elif os.path.exists('/tmp'):
        # Fallback to /tmp (not persistent but works)
        LANCEDB_LOCAL_URI = "/tmp/lancedb_data"
        print("☁️ Linux detected - using /tmp for local LanceDB (NOT persistent)")
    else:
        LANCEDB_LOCAL_URI = "./lancedb_data"
else:
    # Local Windows/Mac development
    LANCEDB_LOCAL_URI = "./lancedb_data"

# Remote LanceDB Cloud (for production/backend access)
LANCEDB_REMOTE_URI = "db://vqaproject-p9vmdg"
LANCEDB_API_KEY = os.environ["LANCEDB_API_KEY"]  # set via environment variable, do not hardcode

# Table name for VQA embeddings (same for both)
TABLE_NAME = "vqa_embeddings"

# Dataset settings
DATA_DIR = Path("./vqa_data")
USE_VALIDATION_SET = False  # False = training set (~443k samples)
SAMPLE_FRACTION = 0.50      # Use 50% of training set = ~221k samples

# Processing settings
BATCH_SIZE = 32  # Reduce if GPU memory issues (16 for 8GB GPU)
CHECKPOINT_EVERY = 1000  # Save progress every N samples
UPLOAD_BATCH_SIZE = 1000  # LanceDB batch size for uploads

# Embedding settings
EMBEDDING_DIM = 768  # ViLT hidden size
MAX_QUESTION_LENGTH = 40  # Truncate long questions

# ============================================

# Create directories
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Local LanceDB: {LANCEDB_LOCAL_URI}")
print(f"  Remote LanceDB: {LANCEDB_REMOTE_URI}")
print(f"  Table name: {TABLE_NAME}")
print(f"  Data directory: {DATA_DIR.absolute()}")
print(f"  Dataset: {'Validation' if USE_VALIDATION_SET else 'Training'} ({SAMPLE_FRACTION*100:.0f}%)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Embedding dimension: {EMBEDDING_DIM}")

# Estimate size
if USE_VALIDATION_SET:
    estimated_samples = int(214354 * SAMPLE_FRACTION)
else:
    estimated_samples = int(443757 * SAMPLE_FRACTION)
print(f"\n📊 Estimated samples: ~{estimated_samples:,}")
print(f"⏱️ Estimated time: ~{estimated_samples / 32 / 60:.0f} minutes (GPU) or ~{estimated_samples / 8 / 60:.0f} minutes (CPU)")

# Show persistence info
if '/home/azureuser' in LANCEDB_LOCAL_URI:
    print(f"\n✅ Local Lance files will PERSIST after compute restart!")
elif '/tmp' in LANCEDB_LOCAL_URI:
    print(f"\n⚠️ Local Lance files in /tmp will be LOST on restart (use cloud as backup)")

In [ ]:
# Cell 3b: Check for Persistent Storage Options on Azure
# Run this to find if there's persistent storage that supports LanceDB

import subprocess
import os

print("=" * 70)
print("🔍 Checking Azure Storage Options for Persistent LanceDB")
print("=" * 70)

# Check all mounted filesystems
print("\n📁 All Mounted Filesystems:")
try:
    result = subprocess.run(['df', '-Th'], capture_output=True, text=True)
    print(result.stdout)
except:
    print("   Could not get filesystem info")

# Check for common persistent locations
persistent_candidates = [
    "/home/azureuser",           # User home (sometimes persistent)
    "/mnt",                       # Mounted volumes
    "/mnt/batch",                 # Batch storage
    "/mnt/resource",              # Resource disk (NOT persistent usually)
    "/data",                      # Data disk if attached
    "/datadisk",                  # Custom data disk
    os.path.expanduser("~"),      # User home directory
]

print("\n📁 Checking Potential Persistent Locations:")
for path in persistent_candidates:
    if os.path.exists(path):
        # Get filesystem type
        try:
            result = subprocess.run(['df', '-T', path], capture_output=True, text=True)
            lines = result.stdout.strip().split('\n')
            if len(lines) > 1:
                parts = lines[1].split()
                fs_type = parts[1] if len(parts) > 1 else "unknown"
                size = parts[2] if len(parts) > 2 else "?"
                avail = parts[4] if len(parts) > 4 else "?"
                
                # Check if writable
                test_file = os.path.join(path, ".lancedb_test")
                writable = False
                try:
                    os.makedirs(path, exist_ok=True)
                    with open(test_file, 'w') as f:
                        f.write("test")
                    os.remove(test_file)
                    writable = True
                except:
                    pass
                
                # ext4/xfs are good, cifs is bad for LanceDB
                lance_ok = fs_type in ['ext4', 'xfs', 'ext3', 'btrfs']
                
                status = "✅" if (writable and lance_ok) else "⚠️" if writable else "❌"
                print(f"   {status} {path}")
                print(f"      Filesystem: {fs_type}, Size: {size}, Available: {avail}")
                print(f"      Writable: {writable}, LanceDB compatible: {lance_ok}")
        except Exception as e:
            print(f"   ❓ {path} - could not check: {e}")
    else:
        print(f"   ❌ {path} - does not exist")

print("\n" + "-" * 70)
print("ℹ️  Recommendation:")
print("-" * 70)
print("""
   For PERSISTENT LanceDB storage on Azure ML:
   
   1. ✅ USE LANCEDB CLOUD (already configured!)
      - Your data is already in: db://vqaproject-p9vmdg
      - Survives compute restarts
      - Accessible from Flask backend
   
   2. 📁 LOCAL /tmp IS A CACHE ONLY
      - Fast queries during session
      - Gets wiped on restart (normal!)
      - Cell 11 auto-falls back to cloud
   
   3. 🔧 IF YOU NEED PERSISTENT LOCAL FILES:
      - Attach a Premium SSD data disk to your compute
      - Mount it (e.g., /data or /datadisk)
      - Update LANCEDB_LOCAL_URI to point there
""")

In [ ]:
# Cell 4: Checkpoint System

class CheckpointManager:
    """Manage checkpoints to resume interrupted processing."""
    
    def __init__(self, checkpoint_dir: Path = None):
        self.checkpoint_dir = checkpoint_dir or DATA_DIR / "checkpoints"
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_file = self.checkpoint_dir / "progress.json"
    
    def save(self, phase: str, progress: dict):
        """Save checkpoint."""
        checkpoint = {
            "phase": phase,
            "progress": progress,
            "timestamp": datetime.now().isoformat()
        }
        with open(self.checkpoint_file, 'w') as f:
            json.dump(checkpoint, f, indent=2)
        print(f"💾 Checkpoint saved: {phase} - {progress}")
    
    def load(self) -> Optional[dict]:
        """Load checkpoint if exists."""
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file, 'r') as f:
                checkpoint = json.load(f)
            print(f"📂 Loaded checkpoint: {checkpoint['phase']} from {checkpoint['timestamp']}")
            return checkpoint
        return None
    
    def clear(self):
        """Clear checkpoint."""
        if self.checkpoint_file.exists():
            self.checkpoint_file.unlink()
            print("🗑️ Checkpoint cleared")

checkpoint_mgr = CheckpointManager()
print("✓ Checkpoint system initialized")

---
# 📥 Phase 1: Download VQAv2 Dataset

In [ ]:
# Cell 5: Download Helper Functions

def download_file(url: str, destination: Path, description: str = None) -> bool:
    """Download file with progress bar."""
    destination = Path(destination)
    
    if destination.exists():
        print(f"⏭️ Already exists: {destination.name}")
        return True
    
    destination.parent.mkdir(parents=True, exist_ok=True)
    
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(destination, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, 
                     desc=description or destination.name) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
                    pbar.update(len(chunk))
        
        print(f"✓ Downloaded: {destination.name}")
        return True
        
    except Exception as e:
        print(f"✗ Download failed: {e}")
        if destination.exists():
            destination.unlink()
        return False

def extract_zip(zip_path: Path, extract_to: Path) -> bool:
    """Extract zip file."""
    try:
        print(f"📦 Extracting: {zip_path.name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✓ Extracted to: {extract_to}")
        return True
    except Exception as e:
        print(f"✗ Extraction failed: {e}")
        return False

print("✓ Download helpers ready")

In [ ]:
# Cell 6: Download VQAv2 Annotations and Questions

VQA_URLS = {
    "train_annotations": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip",
    "val_annotations": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip",
    "train_questions": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip",
    "val_questions": "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip",
}

COCO_URLS = {
    "train_images": "http://images.cocodataset.org/zips/train2014.zip",
    "val_images": "http://images.cocodataset.org/zips/val2014.zip",
}

print("=" * 60)
print("Downloading VQAv2 Annotations & Questions")
print("=" * 60)

if USE_VALIDATION_SET:
    ann_url = VQA_URLS["val_annotations"]
    que_url = VQA_URLS["val_questions"]
else:
    ann_url = VQA_URLS["train_annotations"]
    que_url = VQA_URLS["train_questions"]

ann_zip = DATA_DIR / "annotations.zip"
que_zip = DATA_DIR / "questions.zip"

download_file(ann_url, ann_zip, "Annotations")
download_file(que_url, que_zip, "Questions")

extract_zip(ann_zip, DATA_DIR)
extract_zip(que_zip, DATA_DIR)

print("\n✓ Annotations and questions ready!")

In [ ]:
# Cell 7: Download COCO Images

print("=" * 60)
print("Downloading COCO Images")
print("=" * 60)
print("⚠️ This will download ~6GB (val) or ~13GB (train) of images\n")

if USE_VALIDATION_SET:
    images_url = COCO_URLS["val_images"]
    images_zip = DATA_DIR / "val2014.zip"
    images_dir = DATA_DIR / "val2014"
else:
    images_url = COCO_URLS["train_images"]
    images_zip = DATA_DIR / "train2014.zip"
    images_dir = DATA_DIR / "train2014"

if images_dir.exists() and len(list(images_dir.glob("*.jpg"))) > 1000:
    print(f"⏭️ Images already exist in {images_dir}")
    print(f"   Found {len(list(images_dir.glob('*.jpg')))} images")
else:
    download_file(images_url, images_zip, "COCO Images")
    extract_zip(images_zip, DATA_DIR)

print("\n✓ Images ready!")

---
# 🔄 Phase 2: Data Preparation

In [ ]:
# Cell 8: Load VQAv2 Dataset

def load_vqa_data(data_dir: Path, use_validation: bool = True) -> Tuple[dict, dict]:
    """Load VQAv2 annotations and questions."""
    
    split = "val" if use_validation else "train"
    
    ann_file = data_dir / f"v2_mscoco_{split}2014_annotations.json"
    que_file = data_dir / f"v2_OpenEnded_mscoco_{split}2014_questions.json"
    
    print(f"Loading {split} dataset...")
    
    with open(ann_file, 'r') as f:
        annotations = json.load(f)
    print(f"  ✓ Loaded {len(annotations['annotations'])} annotations")
    
    with open(que_file, 'r') as f:
        questions = json.load(f)
    print(f"  ✓ Loaded {len(questions['questions'])} questions")
    
    return annotations, questions

annotations_data, questions_data = load_vqa_data(DATA_DIR, USE_VALIDATION_SET)

annotations_lookup = {a['question_id']: a for a in annotations_data['annotations']}
questions_lookup = {q['question_id']: q for q in questions_data['questions']}

print(f"\n✓ Dataset loaded: {len(questions_lookup)} QA pairs")

In [ ]:
# Cell 9: Prepare Dataset for Processing

def get_most_common_answer(annotation: dict) -> Tuple[str, float]:
    """Get most common answer and agreement score."""
    answers = [a['answer'] for a in annotation['answers']]
    counter = Counter(answers)
    most_common = counter.most_common(1)[0]
    return most_common[0], most_common[1] / len(answers)

def get_question_type(question: str) -> str:
    """Classify question type."""
    q = question.lower().strip()
    
    if q.startswith("how many"):
        return "count"
    elif q.startswith("what color"):
        return "color"
    elif q.startswith("what is") or q.startswith("what are"):
        return "what"
    elif q.startswith("where"):
        return "location"
    elif q.startswith("who"):
        return "person"
    elif q.startswith(("is ", "are ", "does ", "do ", "can ", "could ", "was ", "were ")):
        return "yes/no"
    else:
        return "other"

def prepare_samples(questions_lookup: dict, annotations_lookup: dict, 
                   images_dir: Path, sample_fraction: float = 1.0) -> List[dict]:
    """Prepare samples for processing."""
    
    samples = []
    split = "val2014" if USE_VALIDATION_SET else "train2014"
    
    for qid, question_data in tqdm(questions_lookup.items(), desc="Preparing samples"):
        annotation = annotations_lookup.get(qid)
        if not annotation:
            continue
        
        image_id = question_data['image_id']
        image_filename = f"COCO_{split}_{image_id:012d}.jpg"
        image_path = images_dir / image_filename
        
        if not image_path.exists():
            continue
        
        answer, confidence = get_most_common_answer(annotation)
        
        samples.append({
            "question_id": qid,
            "image_id": image_id,
            "question": question_data['question'],
            "answer": answer,
            "answer_confidence": confidence,
            "question_type": get_question_type(question_data['question']),
            "image_path": str(image_path),
            "answer_type": annotation.get('answer_type', 'unknown')
        })
    
    if sample_fraction < 1.0:
        np.random.seed(42)
        n_samples = int(len(samples) * sample_fraction)
        indices = np.random.choice(len(samples), n_samples, replace=False)
        samples = [samples[i] for i in sorted(indices)]
    
    return samples

images_dir = DATA_DIR / ("val2014" if USE_VALIDATION_SET else "train2014")
samples = prepare_samples(questions_lookup, annotations_lookup, images_dir, SAMPLE_FRACTION)

print(f"\n✓ Prepared {len(samples)} samples ({SAMPLE_FRACTION*100:.0f}% of dataset)")

In [ ]:
# Cell 10: Dataset Statistics

print("=" * 60)
print("Dataset Statistics")
print("=" * 60)

qtype_dist = Counter([s['question_type'] for s in samples])
print("\n📊 Question Type Distribution:")
for qtype, count in qtype_dist.most_common():
    pct = count / len(samples) * 100
    print(f"  {qtype:12s}: {count:6d} ({pct:5.1f}%)")

atype_dist = Counter([s['answer_type'] for s in samples])
print("\n📊 Answer Type Distribution:")
for atype, count in atype_dist.most_common():
    pct = count / len(samples) * 100
    print(f"  {atype:12s}: {count:6d} ({pct:5.1f}%)")

print("\n📝 Sample Entries:")
for i, sample in enumerate(samples[:3]):
    print(f"\n  [{i+1}]")
    print(f"  Q: {sample['question']}")
    print(f"  A: {sample['answer']} (confidence: {sample['answer_confidence']:.0%})")

---
# 🗄️ Phase 3: LanceDB Setup

In [ ]:
# Cell 11: Connect to LanceDB (Both Local and Remote)
# Priority: Local first, fallback to Remote Cloud

print("=" * 60)
print("Connecting to LanceDB Instances")
print("=" * 60)

# Connect to LOCAL LanceDB
print("\n📁 Connecting to local LanceDB...")
db_local = lancedb.connect(LANCEDB_LOCAL_URI)
print(f"✓ Connected to local: {LANCEDB_LOCAL_URI}")
local_tables = db_local.table_names()
print(f"  Existing tables: {local_tables}")

# Connect to REMOTE LanceDB Cloud
print("\n☁️ Connecting to remote LanceDB Cloud...")
db_remote = lancedb.connect(LANCEDB_REMOTE_URI, api_key=LANCEDB_API_KEY)
print(f"✓ Connected to remote: {LANCEDB_REMOTE_URI}")
remote_tables = db_remote.table_names()
print(f"  Existing tables: {remote_tables}")

# ============================================
# PREFERENCE: Local first, fallback to Remote
# ============================================
print("\n" + "-" * 60)
print("🔍 Checking for existing table...")

table = None
db = None

# Try LOCAL first
if TABLE_NAME in local_tables:
    print(f"✅ Found '{TABLE_NAME}' in LOCAL LanceDB!")
    print(f"   Using local for faster queries.")
    db = db_local
    table = db_local.open_table(TABLE_NAME)
    print(f"   Table loaded: {len(table)} records")

# Fallback to REMOTE
elif TABLE_NAME in remote_tables:
    print(f"⚠️ Local Lance files not found at: {LANCEDB_LOCAL_URI}")
    print(f"   (This is normal after compute restart - /tmp is ephemeral)")
    print(f"\n☁️ Falling back to REMOTE LanceDB Cloud...")
    db = db_remote
    table = db_remote.open_table(TABLE_NAME)
    print(f"✅ Table loaded from cloud: {len(table)} records")

else:
    print(f"❌ Table '{TABLE_NAME}' not found in local or remote!")
    print(f"   You need to run the embedding generation cells (8-18) first.")

print("-" * 60)

if table is not None:
    print(f"\n✓ Ready to query! Using: {'LOCAL' if db == db_local else 'REMOTE CLOUD'}")
else:
    print(f"\n⚠️ No table loaded - run embedding generation first")

In [ ]:
# Cell 12: Define LanceDB Schema

# Define the schema for VQA embeddings table
# In LanceDB, ALL fields are stored as columns (no separate "metadata" dict)
# Every field is queryable, filterable, and returned in search results
#
# LanceDB Multimodal Support:
# - Can store images/videos directly as binary or PIL Image columns
# - Can store URLs/paths as string columns (lighter weight)
# - We use path approach here to keep storage small

from lancedb.pydantic import LanceModel, Vector
from pydantic import Field
from typing import Optional

class VQAEmbedding(LanceModel):
    """Schema for VQA embeddings in LanceDB.
    
    Note: In LanceDB, all fields ARE the metadata. Unlike Pinecone/Cosmos DB,
    there's no separate metadata dictionary - everything is a first-class column.
    
    For images: We store the path/URL rather than raw bytes to keep storage small.
    LanceDB CAN store raw images (PIL.Image or bytes) if needed for self-contained storage.
    """
    # Identifiers
    question_id: int = Field(description="Unique VQA question ID")
    image_id: int = Field(description="COCO image ID")
    
    # Image reference (path or URL - lightweight approach)
    image_path: Optional[str] = Field(default=None, description="Path or URL to the image")
    
    # Question & Answer (the main metadata we want to retrieve)
    question: str = Field(description="The question text")
    answer: str = Field(description="The ground truth answer")  # ← KEY METADATA
    answer_confidence: float = Field(description="Agreement score among annotators")
    
    # Categorization (for filtering)
    question_type: str = Field(description="Type: count, color, yes/no, etc.")
    answer_type: str = Field(description="Type: yes/no, number, other")
    
    # The embedding vector
    vector: Vector(EMBEDDING_DIM) = Field(description="768-dim ViLT embedding")

print("✓ LanceDB schema defined")
print(f"  Embedding dimension: {EMBEDDING_DIM}")
print(f"\n📋 Schema fields (all are queryable metadata):")
print(f"  • question_id, image_id - Identifiers")
print(f"  • image_path - Reference to original image (URL or path)")
print(f"  • question, answer, answer_confidence - Q&A data")
print(f"  • question_type, answer_type - Categories for filtering")
print(f"  • vector - The 768-dim ViLT embedding")

---
# 🧠 Phase 4: ViLT Embedding Generation

In [ ]:
# Cell 13: Load ViLT Model

print("=" * 60)
print("Loading ViLT Model")
print("=" * 60)

print("Loading processor...")
vilt_processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")

print("Loading model...")
vilt_model = ViltModel.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
vilt_model = vilt_model.to(DEVICE)
vilt_model.eval()

print(f"\n✓ ViLT model loaded on {DEVICE}")
print(f"✓ Embedding dimension: {vilt_model.config.hidden_size}")

In [ ]:
# Cell 14: Embedding Generation Functions

def generate_embedding(image: Image.Image, question: str, 
                      processor, model, device) -> np.ndarray:
    """Generate ViLT embedding for image-question pair."""
    
    inputs = processor(image, question, return_tensors="pt", 
                      truncation=True, max_length=MAX_QUESTION_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    
    embedding = embedding / np.linalg.norm(embedding)
    
    return embedding.flatten()

def generate_embeddings_batch(samples: List[dict], processor, model, 
                             device, batch_size: int = 32) -> List[np.ndarray]:
    """Generate embeddings for a batch of samples."""
    
    embeddings = []
    
    for i in range(0, len(samples), batch_size):
        batch = samples[i:i + batch_size]
        
        images = []
        questions = []
        valid_indices = []
        
        for j, sample in enumerate(batch):
            try:
                img = Image.open(sample['image_path']).convert('RGB')
                images.append(img)
                questions.append(sample['question'])
                valid_indices.append(j)
            except Exception as e:
                print(f"⚠️ Error loading image: {e}")
        
        if not images:
            embeddings.extend([None] * len(batch))
            continue
        
        try:
            inputs = processor(images, questions, return_tensors="pt", 
                             padding=True, truncation=True, 
                             max_length=MAX_QUESTION_LENGTH)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            norms = np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
            batch_embeddings = batch_embeddings / norms
            
            result = [None] * len(batch)
            for idx, emb in zip(valid_indices, batch_embeddings):
                result[idx] = emb
            embeddings.extend(result)
            
        except Exception as e:
            print(f"⚠️ Batch processing error: {e}")
            embeddings.extend([None] * len(batch))
    
    return embeddings

print("✓ Embedding functions ready")

In [ ]:
# Cell 15: Test Embedding Generation

print("Testing embedding generation...")

test_sample = samples[0]
test_image = Image.open(test_sample['image_path']).convert('RGB')

start_time = time.time()
test_embedding = generate_embedding(
    test_image, test_sample['question'], 
    vilt_processor, vilt_model, DEVICE
)
elapsed = time.time() - start_time

print(f"\n✓ Test embedding generated")
print(f"  Shape: {test_embedding.shape}")
print(f"  Dtype: {test_embedding.dtype}")
print(f"  Norm: {np.linalg.norm(test_embedding):.4f} (should be ~1.0)")
print(f"  Time: {elapsed:.3f}s")
print(f"\n  Question: {test_sample['question']}")
print(f"  Answer: {test_sample['answer']}")

In [ ]:
# Cell 16: Generate All Embeddings

print("=" * 60)
print(f"Generating Embeddings for {len(samples)} Samples")
print("=" * 60)

embeddings_cache_file = DATA_DIR / "embeddings_cache.pkl"
start_idx = 0
all_embeddings = []

if embeddings_cache_file.exists():
    with open(embeddings_cache_file, 'rb') as f:
        cache = pickle.load(f)
    all_embeddings = cache['embeddings']
    start_idx = cache['last_idx']
    print(f"📂 Resuming from checkpoint: {start_idx}/{len(samples)}")

remaining_samples = samples[start_idx:]
total_batches = (len(remaining_samples) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Processing {len(remaining_samples)} remaining samples in {total_batches} batches...\n")

for batch_idx in tqdm(range(0, len(remaining_samples), BATCH_SIZE), 
                      desc="Generating embeddings", total=total_batches):
    
    batch = remaining_samples[batch_idx:batch_idx + BATCH_SIZE]
    
    batch_embeddings = generate_embeddings_batch(
        batch, vilt_processor, vilt_model, DEVICE, BATCH_SIZE
    )
    all_embeddings.extend(batch_embeddings)
    
    current_idx = start_idx + batch_idx + len(batch)
    if (batch_idx + BATCH_SIZE) % CHECKPOINT_EVERY < BATCH_SIZE:
        with open(embeddings_cache_file, 'wb') as f:
            pickle.dump({
                'embeddings': all_embeddings,
                'last_idx': current_idx
            }, f)
        print(f"\n💾 Checkpoint: {current_idx}/{len(samples)}")

with open(embeddings_cache_file, 'wb') as f:
    pickle.dump({
        'embeddings': all_embeddings,
        'last_idx': len(samples)
    }, f)

print(f"\n✓ Generated {len(all_embeddings)} embeddings")
print(f"✓ Failed: {sum(1 for e in all_embeddings if e is None)}")

---
# ⬆️ Phase 5: Upload to LanceDB

In [ ]:
# Cell 17: Prepare Documents for Upload

def prepare_lancedb_records(samples: List[dict], embeddings: List[np.ndarray]) -> List[dict]:
    """Prepare records for LanceDB insertion."""
    
    records = []
    skipped = 0
    
    for sample, embedding in zip(samples, embeddings):
        if embedding is None:
            skipped += 1
            continue
        
        records.append({
            "question_id": sample['question_id'],
            "image_id": sample['image_id'],
            "image_path": sample['image_path'],  # ← Now storing image path!
            "question": sample['question'],
            "answer": sample['answer'],
            "answer_confidence": sample['answer_confidence'],
            "question_type": sample['question_type'],
            "answer_type": sample['answer_type'],
            "vector": embedding.tolist()
        })
    
    return records, skipped

records, skipped = prepare_lancedb_records(samples, all_embeddings)

print(f"✓ Prepared {len(records)} records for upload")
print(f"✓ Skipped {skipped} failed embeddings")
print(f"✓ Each record includes: image_path for reference")

In [ ]:
# Cell 17b: DEBUG - Check Local Path
# Run this cell to see exactly what path LanceDB is trying to use

print("=" * 70)
print("🔍🔍🔍 DEBUG - WHERE IS LOCAL LANCEDB TRYING TO SAVE? 🔍🔍🔍")
print("=" * 70)

print(f"\n1. LANCEDB_LOCAL_URI variable = '{LANCEDB_LOCAL_URI}'")
print(f"2. Absolute path = '{Path(LANCEDB_LOCAL_URI).absolute()}'")
print(f"3. Does path exist? {Path(LANCEDB_LOCAL_URI).exists()}")
print(f"4. Does parent exist? {Path(LANCEDB_LOCAL_URI).parent.exists()}")

print(f"\n5. Platform detected: {platform.system()}")
print(f"6. Does /tmp exist? {os.path.exists('/tmp')}")

# Try to get filesystem info
print("\n7. Filesystem type:")
try:
    import subprocess
    result = subprocess.run(['df', '-T', str(Path(LANCEDB_LOCAL_URI).parent)], 
                          capture_output=True, text=True, timeout=5)
    print(result.stdout if result.stdout else "   (no output)")
    if result.stderr:
        print(f"   stderr: {result.stderr}")
except Exception as e:
    print(f"   Could not determine: {e}")

# Try to get mount info
print("\n8. Mount info for path:")
try:
    result = subprocess.run(['findmnt', '-T', str(Path(LANCEDB_LOCAL_URI).parent)], 
                          capture_output=True, text=True, timeout=5)
    print(result.stdout if result.stdout else "   (no output)")
except Exception as e:
    print(f"   Could not determine: {e}")

print("\n" + "=" * 70)

In [ ]:
# Cell 18: Upload to LanceDB (BOTH Local and Remote) - APPEND MODE

# ============================================
# DEBUG: Show the actual paths being used FIRST
# ============================================
print("=" * 70)
print("🔍🔍🔍 DEBUG - LOCAL PATH INFO 🔍🔍🔍")
print("=" * 70)
print(f"   LANCEDB_LOCAL_URI = {LANCEDB_LOCAL_URI}")
print(f"   Absolute path: {Path(LANCEDB_LOCAL_URI).absolute()}")
print(f"   Path exists: {Path(LANCEDB_LOCAL_URI).exists()}")
print(f"   Parent exists: {Path(LANCEDB_LOCAL_URI).parent.exists()}")

# Check filesystem type (for debugging)
import subprocess
try:
    result = subprocess.run(['df', '-T', str(Path(LANCEDB_LOCAL_URI).parent)], 
                          capture_output=True, text=True)
    print(f"   Filesystem info:\n{result.stdout}")
except Exception as e:
    print(f"   (Could not get filesystem info: {e})")

# Also show what Cell 3 detected
print(f"\n   Platform: {platform.system()}")
print(f"   /tmp exists: {os.path.exists('/tmp')}")
print("=" * 70)

# Upload to BOTH local and remote LanceDB - APPEND to existing data!

def upload_to_lancedb(db, db_name: str, table_name: str, records: List[dict], 
                      batch_size: int = 1000, mode: str = "append") -> Optional:
    """Upload records to a LanceDB instance.
    
    Args:
        mode: "append" = add to existing table, "replace" = drop and recreate
    """
    print(f"\n{'📁' if 'local' in db_name.lower() else '☁️'} Uploading to {db_name}...")
    print(f"   Mode: {mode.upper()}")
    
    try:
        # Check if table exists
        existing_tables = db.table_names() if hasattr(db, 'table_names') else []
        
        if table_name in existing_tables:
            if mode == "replace":
                print(f"  ⚠️ Table '{table_name}' exists - dropping and recreating")
                db.drop_table(table_name)
                # Create new table
                print(f"  Creating table '{table_name}'...")
                first_batch = records[:batch_size]
                table = db.create_table(table_name, data=first_batch, schema=VQAEmbedding)
                remaining = records[batch_size:]
            else:  # append mode
                print(f"  ✅ Table '{table_name}' exists - APPENDING {len(records):,} new records")
                table = db.open_table(table_name)
                existing_count = len(table)
                print(f"     Existing records: {existing_count:,}")
                remaining = records  # Add all records
        else:
            # Create new table
            print(f"  Creating new table '{table_name}'...")
            first_batch = records[:batch_size]
            table = db.create_table(table_name, data=first_batch, schema=VQAEmbedding)
            remaining = records[batch_size:]
        
        # Add remaining batches
        if remaining:
            for i in tqdm(range(0, len(remaining), batch_size), 
                         desc=f"  Uploading to {db_name}"):
                batch = remaining[i:i + batch_size]
                table.add(batch)
        
        final_count = len(table)
        print(f"  ✓ Added {len(records):,} records")
        print(f"  ✓ Total rows now: {final_count:,}")
        return table
        
    except Exception as e:
        print(f"  ✗ Error uploading to {db_name}: {e}")
        return None

print("\n" + "=" * 60)
print("Uploading to BOTH Local and Remote LanceDB (APPEND MODE)")
print("=" * 60)
print(f"📊 New records to add: {len(records):,}")

# ============================================
# Upload to LOCAL LanceDB (APPEND)
# ============================================
print("\n--- LOCAL LANCEDB ---")
print(f"   Target path: {LANCEDB_LOCAL_URI}")
try:
    table_local = upload_to_lancedb(
        db_local, "Local LanceDB", TABLE_NAME, records, UPLOAD_BATCH_SIZE, mode="append"
    )
except Exception as e:
    print(f"  ✗ Local upload failed: {e}")
    table_local = None

# ============================================
# Upload to REMOTE LanceDB Cloud (APPEND)
# ============================================
print("\n--- REMOTE LANCEDB CLOUD ---")
table_remote = upload_to_lancedb(
    db_remote, "Remote LanceDB Cloud", TABLE_NAME, records, UPLOAD_BATCH_SIZE, mode="append"
)

# Use whichever table is available for queries (prefer local for speed)
table = table_local if table_local else table_remote

print("\n" + "=" * 60)
print("✓ Upload Complete!")
print("=" * 60)
print(f"  📁 Local:  {len(table_local):,} total records" if table_local else "  📁 Local:  Failed")
print(f"  ☁️ Remote: {len(table_remote):,} total records" if table_remote else "  ☁️ Remote: Failed")

if table_local and table_remote:
    print("\n✅ Both local and remote updated successfully!")
elif table_remote:
    print("\n⚠️ Only remote saved (local failed - may be filesystem issue)")
elif table_local:
    print("\n⚠️ Only local saved (remote failed - check connection)")
else:
    print("\n❌ Both uploads failed!")

In [ ]:
# Cell 19: Verify Upload & Create Index

print("=" * 60)
print("Verifying Upload")
print("=" * 60)

# Use remote table (works in cloud environments)
# table is already set from Cell 18
if table is None:
    print("❌ No table available - upload may have failed")
else:
    print(f"\n📊 Total rows: {len(table)}")
    
    # Sample record
    sample_df = table.head(1).to_pandas()
    print(f"\n📝 Sample Record:")
    print(f"  Question ID: {sample_df['question_id'].iloc[0]}")
    print(f"  Question: {sample_df['question'].iloc[0]}")
    print(f"  Answer: {sample_df['answer'].iloc[0]}")
    print(f"  Image Path: {sample_df['image_path'].iloc[0][:50]}...")
    print(f"  Vector dim: {len(sample_df['vector'].iloc[0])}")
    
    # Create vector index for faster search (optional, improves performance for large datasets)
    print("\n🔧 Creating vector index...")
    try:
        table.create_index(
            metric="cosine",
            num_partitions=256,
            num_sub_vectors=96
        )
        print("✓ Vector index created")
    except Exception as e:
        # Index may already exist or not be supported
        print(f"⚠️ Index note: {e}")
    
    checkpoint_mgr.save("upload_complete", {
        "total_records": len(table),
        "table_name": TABLE_NAME,
        "storage": "remote" if table == table_remote else "local",
        "timestamp": datetime.now().isoformat()
    })

In [ ]:
# Cell 20: Explore LanceDB Files (Native .lance Format)

# LanceDB uses the Lance format (.lance) which is specifically designed for:
# - Vector similarity search with efficient indexing
# - Columnar storage optimized for ML workloads
# - Fast random access and filtering
# - NOT Parquet - Lance is a separate format built for vector DBs

from pathlib import Path

print("=" * 60)
print("📁 Exploring LanceDB Local Files (.lance format)")
print("=" * 60)

# Check what's in the local LanceDB directory
lance_path = Path(LANCEDB_LOCAL_URI)

if lance_path.exists():
    print(f"\n✓ LanceDB directory: {lance_path.absolute()}")
    
    # List all files recursively
    print(f"\n📂 Lance File Structure:")
    lance_files = []
    other_files = []
    
    for item in sorted(lance_path.rglob("*")):
        if item.is_file():
            size_kb = item.stat().st_size / 1024
            rel_path = item.relative_to(lance_path)
            
            # Categorize files
            if '.lance' in str(item) or item.suffix in ['.lance', '.idx', '.manifest']:
                lance_files.append((rel_path, size_kb))
                print(f"   🔷 {rel_path} ({size_kb:.1f} KB)")
            else:
                other_files.append((rel_path, size_kb))
                print(f"   📄 {rel_path} ({size_kb:.1f} KB)")
        elif item.is_dir():
            rel_path = item.relative_to(lance_path)
            if '.lance' in str(item):
                print(f"   🔷 {rel_path}/ (Lance table)")
            else:
                print(f"   📁 {rel_path}/")
    
    # Show totals
    total_size = sum(f.stat().st_size for f in lance_path.rglob("*") if f.is_file())
    print(f"\n   📊 Total Lance storage: {total_size / 1024 / 1024:.2f} MB")
    print(f"   📊 Lance files: {len(lance_files)}")
    
    # Explain the Lance format
    print("\n" + "-" * 60)
    print("ℹ️  About Lance Format:")
    print("-" * 60)
    print("""
    Lance (.lance) is a columnar format designed for ML & vector search:
    
    ✓ Native vector indexing (IVF, HNSW)
    ✓ Optimized for similarity search
    ✓ Fast random access for embeddings
    ✓ Efficient filtering with predicates
    ✓ Versioned & ACID compliant
    
    ❌ NOT Parquet - Parquet has no vector indexing
    ❌ NOT for traditional SQL databases
    
    These .lance files contain:
    - Vector embeddings (768-dim ViLT)
    - Metadata columns (question, answer, etc.)
    - Vector index structures
    """)

else:
    print(f"\n⚠️ LanceDB directory not found: {lance_path}")
    print("   Local save may have failed (Azure filesystem issue)")
    print("   Your data is still safe in LanceDB Cloud!")

# Show the table info from LanceDB directly
print("\n" + "=" * 60)
print("📊 LanceDB Table Info")
print("=" * 60)

if table is not None:
    print(f"\n   Table name: {TABLE_NAME}")
    print(f"   Total records: {len(table)}")
    print(f"   Vector dimension: {EMBEDDING_DIM}")
    
    # Show schema
    print(f"\n   Schema columns:")
    sample = table.head(1).to_pandas()
    for col in sample.columns:
        dtype = sample[col].dtype
        print(f"      • {col}: {dtype}")
else:
    print("\n   ❌ No table loaded")

---
# 🔍 Phase 6: Query System

In [ ]:
# Cell 20: Vector Search Function

def vector_search(table, query_embedding: np.ndarray, k: int = 5,
                 filter_query: str = None) -> pd.DataFrame:
    """Perform vector similarity search in LanceDB."""
    
    # Build query
    query = table.search(query_embedding.tolist()).limit(k)
    
    # Add filter if provided
    if filter_query:
        query = query.where(filter_query)
    
    # Execute and return results
    results = query.to_pandas()
    
    return results

print("✓ Vector search function ready")

In [ ]:
# Cell 21: Query Interface

class VQAQuerySystem:
    """Interactive VQA query system with LanceDB."""
    
    def __init__(self, table, processor, model, device):
        self.table = table
        self.processor = processor
        self.model = model
        self.device = device
    
    def query(self, image_path: str, question: str, k: int = 5,
             filter_type: str = None) -> dict:
        """Query the VQA system with an image and question."""
        
        image = Image.open(image_path).convert('RGB')
        
        start_time = time.time()
        query_embedding = generate_embedding(
            image, question, self.processor, self.model, self.device
        )
        embed_time = time.time() - start_time
        
        filter_query = f"question_type = '{filter_type}'" if filter_type else None
        start_time = time.time()
        results_df = vector_search(self.table, query_embedding, k, filter_query)
        search_time = time.time() - start_time
        
        if len(results_df) > 0:
            answers = results_df['answer'].tolist()
            counter = Counter(answers)
            predicted_answer = counter.most_common(1)[0][0]
            confidence = counter.most_common(1)[0][1] / len(answers)
        else:
            predicted_answer = "unknown"
            confidence = 0.0
        
        return {
            "question": question,
            "predicted_answer": predicted_answer,
            "confidence": confidence,
            "similar_examples": results_df.to_dict('records'),
            "embedding_time_ms": embed_time * 1000,
            "search_time_ms": search_time * 1000
        }
    
    def display_results(self, result: dict):
        """Display query results nicely."""
        print("\n" + "=" * 60)
        print("🔍 QUERY RESULTS")
        print("=" * 60)
        print(f"\n❓ Question: {result['question']}")
        print(f"\n🎯 Predicted Answer: {result['predicted_answer']}")
        print(f"   Confidence: {result['confidence']:.0%}")
        print(f"\n⏱️ Embedding time: {result['embedding_time_ms']:.1f}ms")
        print(f"   Search time: {result['search_time_ms']:.1f}ms")
        
        print(f"\n📋 Similar Examples:")
        for i, ex in enumerate(result['similar_examples'][:5], 1):
            score = ex.get('_distance', 0)
            print(f"\n   [{i}] Distance: {score:.4f}")
            print(f"       Q: {ex['question']}")
            print(f"       A: {ex['answer']}")

# Initialize query system
query_system = VQAQuerySystem(
    table, vilt_processor, vilt_model, DEVICE
)

print("✓ Query system initialized")

In [ ]:
# Cell 22: Test Query

test_sample = samples[100]

print(f"Testing with image: {test_sample['image_path']}")
print(f"Ground truth answer: {test_sample['answer']}")

result = query_system.query(
    image_path=test_sample['image_path'],
    question=test_sample['question'],
    k=5
)

query_system.display_results(result)

match = result['predicted_answer'].lower() == test_sample['answer'].lower()
print(f"\n{'✓ CORRECT!' if match else '✗ INCORRECT'}")
print(f"Ground truth: {test_sample['answer']}")

In [ ]:
# Cell 23: Interactive Query

# ============================================
# 🎮 TRY YOUR OWN QUERIES HERE
# ============================================

test_image_path = samples[0]['image_path']
my_question = "What color is the object?"

result = query_system.query(
    image_path=test_image_path,
    question=my_question,
    k=5
)

query_system.display_results(result)

from IPython.display import display
img = Image.open(test_image_path)
img.thumbnail((400, 400))
display(img)

---
# 📊 Phase 7: Evaluation

In [ ]:
# Cell 24: Evaluation Functions

def evaluate_vqa_accuracy(query_system, test_samples: List[dict], 
                         k: int = 5) -> dict:
    """Evaluate VQA accuracy on test samples."""
    
    correct = 0
    total = 0
    results_by_type = {}
    all_results = []
    
    for sample in tqdm(test_samples, desc="Evaluating"):
        try:
            result = query_system.query(
                image_path=sample['image_path'],
                question=sample['question'],
                k=k
            )
            
            predicted = result['predicted_answer'].lower().strip()
            ground_truth = sample['answer'].lower().strip()
            is_correct = predicted == ground_truth
            
            total += 1
            if is_correct:
                correct += 1
            
            qtype = sample['question_type']
            if qtype not in results_by_type:
                results_by_type[qtype] = {'correct': 0, 'total': 0}
            results_by_type[qtype]['total'] += 1
            if is_correct:
                results_by_type[qtype]['correct'] += 1
            
            all_results.append({
                'question': sample['question'],
                'predicted': predicted,
                'ground_truth': ground_truth,
                'correct': is_correct,
                'question_type': qtype,
                'confidence': result['confidence']
            })
            
        except Exception as e:
            print(f"\n⚠️ Error: {e}")
    
    accuracy = correct / total if total > 0 else 0
    
    type_accuracies = {}
    for qtype, data in results_by_type.items():
        type_accuracies[qtype] = data['correct'] / data['total'] if data['total'] > 0 else 0
    
    return {
        'overall_accuracy': accuracy,
        'correct': correct,
        'total': total,
        'accuracy_by_type': type_accuracies,
        'counts_by_type': results_by_type,
        'all_results': all_results
    }

print("✓ Evaluation functions ready")

In [ ]:
# Cell 25: Run Evaluation

EVAL_SIZE = 500

np.random.seed(42)
eval_indices = np.random.choice(len(samples), min(EVAL_SIZE, len(samples)), replace=False)
eval_samples = [samples[i] for i in eval_indices]

print(f"Evaluating on {len(eval_samples)} samples...\n")

eval_results = evaluate_vqa_accuracy(query_system, eval_samples, k=5)

print("\n" + "=" * 60)
print("📊 EVALUATION RESULTS")
print("=" * 60)
print(f"\n🎯 Overall Accuracy: {eval_results['overall_accuracy']:.1%}")
print(f"   ({eval_results['correct']}/{eval_results['total']} correct)")

print("\n📊 Accuracy by Question Type:")
for qtype, acc in sorted(eval_results['accuracy_by_type'].items(), 
                         key=lambda x: x[1], reverse=True):
    count = eval_results['counts_by_type'][qtype]['total']
    print(f"   {qtype:12s}: {acc:5.1%} ({count} samples)")

In [ ]:
# Cell 26: Save Evaluation Results

eval_results_file = DATA_DIR / "evaluation_results_lancedb.json"

results_to_save = {
    'overall_accuracy': eval_results['overall_accuracy'],
    'correct': eval_results['correct'],
    'total': eval_results['total'],
    'accuracy_by_type': eval_results['accuracy_by_type'],
    'counts_by_type': eval_results['counts_by_type'],
    'eval_size': len(eval_samples),
    'k': 5,
    'database': 'LanceDB',
    'timestamp': datetime.now().isoformat()
}

with open(eval_results_file, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"✓ Results saved to {eval_results_file}")

---
# 🧹 Phase 8: Utilities & Cleanup

In [ ]:
# Cell 27: Utility Functions

def get_table_stats(table):
    """Get table statistics."""
    df = table.to_pandas()
    
    stats = {
        'total_rows': len(df),
        'question_types': df['question_type'].value_counts().to_dict(),
        'answer_types': df['answer_type'].value_counts().to_dict()
    }
    
    return stats

def delete_table(db, table_name: str, confirm: bool = False):
    """Delete a table from LanceDB."""
    if not confirm:
        print("⚠️ Set confirm=True to actually delete")
        return
    
    if table_name in db.table_names():
        db.drop_table(table_name)
        print(f"✓ Deleted table: {table_name}")
    else:
        print(f"⚠️ Table not found: {table_name}")

# Display current stats
stats = get_table_stats(table)
print("📊 Table Statistics:")
print(f"   Total rows: {stats['total_rows']}")
print(f"   Question types: {len(stats['question_types'])}")

In [ ]:
# Cell 28: Cleanup (Run when done)

# ============================================
# 🧹 CLEANUP
# ============================================

# Clear checkpoints
# checkpoint_mgr.clear()

# Delete cached embeddings
# if embeddings_cache_file.exists():
#     embeddings_cache_file.unlink()
#     print("✓ Embeddings cache deleted")

# Delete LanceDB table (CAREFUL!)
# delete_table(db, TABLE_NAME, confirm=True)

# Clear GPU memory
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     print("✓ GPU memory cleared")

print("""ℹ️ Cleanup options available:
- Uncomment checkpoint_mgr.clear() to clear checkpoints
- Uncomment embeddings_cache_file.unlink() to delete embeddings cache
- Uncomment delete_table() to delete LanceDB table
- Uncomment torch.cuda.empty_cache() to free GPU memory
""")

print("\n" + "=" * 60)
print("🎉 NOTEBOOK COMPLETE!")
print("=" * 60)
print(f"\n✅ Embeddings: {len(all_embeddings)} generated")
print(f"✅ Records: {len(table)} in LanceDB")
print(f"✅ Evaluation: {eval_results['overall_accuracy']:.1%} accuracy")
print(f"\n🚀 Your VQA vector search system with LanceDB is ready!")